# Unit 14 exercises: the full system (what earns its place)

Run the recsys data generators **and** the model-cache build (`build_models.py`, the lesson's first cell names the command) before you begin. The lesson assembled one recommender from everything in the book: eight retrieval paths, a calibrated blend into a 50-book candidate pool, and a learned reranker on top, plus the self-attention sequence model on the session log. Its honest result was that **no system dominates**. The full stack (about **0.356 hit@10**) sits just below the fixed 8-path blend (about **0.362**), and the oldest build, the Part-1 blend, still reaches furthest into the catalog. These exercises make you **use** every heavy piece of Part 2 inside that system and measure what it buys: the **HNSW index** (an `efSearch` / `M` sweep, recall vs time), the **feature tower** on cold books (a cold-item slice through the whole stack), **hard negatives** (the cached negative-pool ablation), the **learned reranker** (a feature-family mask and refit on its cached training matrix), and **self-attention** (a SASRec readout on the session log). Then you read the **ablation table**, the **beyond-accuracy table**, the Part-1 benchmark, and the hit-vs-reach **frontier**.

Everything is measured on the **validation** split (`val`). The main-log readouts use the scoreboard's **val cohort**: the 500 warm readers with at least one unseen val positive (cold readers excluded); each reader's **seen** set is their train-positive history. The sequence path is scored on the **session log** only (its own val cohort of 1,419 readers), so its numbers are never compared with the main log's. The sealed `test` holdout stays sealed -- no cell here may touch it. Each answer is left for you to complete; no solutions live here.

A speed note. **Nothing expensive is fitted in this notebook.** MF, the two-tower, both feature towers, the HNSW index's source vectors, both rerankers and SASRec all load from the cache in about a second, and the cheap paths (popularity, lexical, item-item, semantic) refit in seconds. Do **not** fit an MF, a two-tower, a feature tower or a SASRec model anywhere here. The only training you do is the reranker's tiny MLP, refit on the **cached** training matrix (under a second per refit). The first full-stack readout takes about 20-25 s, because it asks all eight paths for every reader's top-50 once; `FullSystem` memoizes those answers, so every later blend or reranker readout over the same pool takes a second or two. The whole notebook runs in about 75 s.

In [ ]:
import time

import numpy as np
import pandas as pd
from bookrec import (
    AnnRetrievalPath,
    FixedBlendPath,
    FullSystem,
    HnswIndex,
    ItemItemRetrievalPath,
    NeuralRerankerPath,
    RerankerModel,
    SasrecRetrievalPath,
    catalog_coverage,
    load_book_data,
    load_cached_models,
)
from bookrec.rerank import family_mask, feature_names
from bookrec.system import FULL8_PATH_NAMES, top_k

SEED = 0         # every cached model was trained with seed 0; the reranker refits below use it too
K = 10           # every readout is at the top-10 cutoff
EF_SEARCH = 64   # the served HNSW index's pinned efSearch (Unit 10)

# --- the generated data and the model cache (loads only; no expensive refit) ----------------
data = load_book_data()                              # catalog, keywords, GloVe, main log, val cohort
cache = load_cached_models(data_dir=data.data_dir)   # every expensive model's artifact + readouts.json
system = FullSystem.from_cache(cache, data)          # cheap paths refit in seconds; the rest loaded
paths = system.paths                                 # {name: fitted path}, e.g. "two-tower", "ann", "feature-tower"
cohort = data.cohort                                 # the 500-reader main-log val cohort (ascending ids)

# Books with NO train positive (818 of the 2,000) -- the cold-item slice of Exercises 2, 3 and 9.
zero_train = set(data.catalog_ids) - set(paths["popularity"].counts)

# system.top_lists(path)  -> {reader: top-10 item ids} over the cohort (seen books excluded)
# system.readout(lists)   -> {"hit", "coverage", "head_share", "exposure_gini", "author_gini",
#                             "novelty", "diversity"} of such a mapping
# system.evaluate(path)   -> system.readout(system.top_lists(path))
# system.full_stack       -> the 8-path pool + learned reranker (two-tower served through HNSW)
# system.fixed_blend(drop=None) -> the equal-weight 8-path fixed blend (drop one path for leave-one-out)
# system.full8            -> the 8 serving paths, {name: path}, in FULL8_PATH_NAMES order


def cold_in(lists):
    # The DISTINCT zero-train books that appear anywhere in a {reader: top-10} mapping.
    return {item for top in lists.values() for item in top} & zero_train

## Exercise 1 - The served HNSW index: an `efSearch` and `M` sweep (recall vs time)

The full stack never scores the two-tower by brute force: it serves it through the Unit-10 **HNSW** index (`paths["ann"]`). ANN is a **speed** tool, so the question to ask of it is a fidelity question: does the approximate search return the same top-10 the exact scan would? The measure is **recall@10 vs exact**: per reader, the fraction of the exact top-10 that the ANN search also returns, averaged over the 500 cohort readers. Two knobs control it. `efSearch` is the **query-time** breadth (how much of the graph each search explores). `M` is the **build-time** degree (how many neighbours each node links to); a bigger `M` makes a denser, better-connected graph.

The scaffold gives the exact reference: the cached index's item matrix, the cohort readers' two-tower vectors, and each reader's exact top-10 rows by the same inner product. Write `recall_and_time(index, ef)`: one **batched** `index.search(queries, K, efSearch=ef)` over all 500 readers, timed with `time.perf_counter()`, returning `(mean recall@10, seconds)`. Sweep the served index over `efSearch` in `1, 4, 16, 64, 256`. Expect recall about **0.733, 0.933, 0.996, 0.999, 1.000**, and the time to grow with `efSearch` (a few milliseconds up to tens of milliseconds -- machine-dependent, so read the shape and do not gate on it). Then build three fresh indexes, `HnswIndex(item_matrix, item_ids, m=M)` for `M` in `4, 8, 32` (about 0.2 s each), and read their recall at `efSearch=16`: about **0.845, 0.953, 0.996**.

Finally, does recall move the *system's* accuracy? Load a **separate** ANN path from the cache for the sweep (never change the served `paths["ann"]` -- the full stack's memo uses it), set its `efSearch` to 4 and then to 64, and score hit@10 with `system.evaluate`. Expect **0.346** at `efSearch=4` and **0.340** at 64 -- the exact two-tower's own hit. Gate the recall trends on their **endpoints** only (recall plateaus near 1.0). In three or four sentences, explain what each knob trades, why the served value of 64 is a sensible choice here, and why a 0.006 *higher* hit at `efSearch=4` is not a reason to serve it.

In [ ]:
# Given: the exact brute-force reference, from the cached ANN artifact (the two-tower's vectors).
ann_state = cache.artifact("ann")
item_matrix = np.array(ann_state["index"]["items"], dtype=np.float32)   # (2000, dim) item vectors
item_ids = list(ann_state["index"]["ids"])                              # aligned to the index rows
row_of = {reader: i for i, reader in enumerate(ann_state["reader_ids"])}
queries = np.array(ann_state["reader_embeddings"], dtype=np.float32)[[row_of[r] for r in cohort]]
exact_rows = np.argsort(-(queries @ item_matrix.T), axis=1)[:, :K]       # exact top-10 ROW indices


def recall_and_time(index, ef):
    # (mean recall@10 vs exact over the cohort, seconds for ONE batched search at efSearch=ef).
    return None   # TODO: start = time.perf_counter(); labels, _ = index.search(queries, K, efSearch=ef);
                  #       seconds = time.perf_counter() - start; per reader i, the overlap
                  #       len(set(labels[i].tolist()) & set(exact_rows[i].tolist())) / K; return (mean, seconds)


served_index = paths["ann"].index   # the full stack's HNSW index (M = 32, efSearch pinned at 64)
EF_SWEEP = [1, 4, 16, 64, 256]
ef_table = None        # TODO: pd.DataFrame, one row per ef in EF_SWEEP: "efSearch", "recall", "seconds"
                       #       -- recall ~0.733, 0.933, 0.996, 0.999, 1.000; seconds grow with efSearch
recall_rises = None    # TODO: bool - recall at efSearch=1 < recall at efSearch=256   (endpoints only)
pinned_recall = None   # TODO: the recall at EF_SEARCH (64)   -- ~0.999
pinned_is_faithful = None   # TODO: bool - pinned_recall >= 0.995
slower_when_broader = None  # TODO: bool - seconds at 256 > seconds at 1   (record it; timing is machine-dependent)

M_SWEEP = [4, 8, 32]
m_table = None         # TODO: pd.DataFrame, one row per M: "M", "recall_ef16" -- recall_and_time(
                       #       HnswIndex(item_matrix, item_ids, m=M), 16)[0]   -- ~0.845, 0.953, 0.996
m_recall_rises = None  # TODO: bool - recall at M=4 < recall at M=32   (endpoints only)

sweep_ann = AnnRetrievalPath(paths["two-tower"]).load(ann_state)   # a SEPARATE copy for the sweep
ann_hit = None         # TODO: {ef: hit@10} for ef in (4, 64): set sweep_ann.efSearch = ef, then
                       #       system.evaluate(sweep_ann)["hit"]   -- ~0.346 at 4, ~0.340 at 64
exact_hit = None       # TODO: system.evaluate(paths["two-tower"])["hit"]   -- ~0.340 (the exact scan)
pinned_matches_exact = None   # TODO: bool - abs(ann_hit[64] - exact_hit) <= 0.002
hit_moves_little = None       # TODO: bool - abs(ann_hit[4] - ann_hit[64]) <= 0.01

ann_explanation = "TODO"  # 3-4 sentences: efSearch trades query time for recall at query time, M trades
                          # build time/memory for a better-connected graph; at 64 the index returns
                          # essentially the exact top-10 (recall ~0.999) for a few ms per 500 readers;
                          # recall measures FIDELITY to the exact scan -- a lower-recall index reorders
                          # near-ties, and hit@10 on 500 readers moves by a few readers either way, so a
                          # 0.006 "gain" is noise, not a reason to serve a less faithful index

## Exercise 2 - A cold-item slice: does the feature tower's cold reach survive the stack?

The Unit-9 **feature tower** builds every book's vector from its *features* (genres, author, keyword embedding), so it can score a book nobody has read yet -- the 818 **zero-train** books. The ID-only two-tower cannot. The full stack serves the feature tower (warm negatives, as shipped) as one of its eight paths, so the question for the system is: does that cold reach **survive** the blend and the reranker?

First, reproduce the Unit-9 readout with the cached towers. Write `cold_coverage(path)`: score every *fitted*, non-cold reader of the path (its `artifact()["reader_ids"]`) from their train history and count the distinct zero-train books in their top-10s (Unit 9's validation-free cohort). Expect **113** of 818 for the feature tower and exactly **0** for the two-tower.

Then follow the cold books through the system on the 500-reader cohort, using `cold_in`. Expect the feature tower's own lists to surface **109** distinct zero-train books; the fixed 8-path blend only **23** (in 49 of its 5,000 slots), of which just **15** are among the feature tower's 109; the fixed blend **without** the feature tower (`system.fixed_blend(drop="feature-tower")`) **27**; and the full stack (the reranker over the same pool) **0**. In three or four sentences, explain where the blend's cold books come from (which paths can score a zero-train book at all?), why removing the feature tower does not lower the count, why the reranker shows none, and what a system that *must* serve new books would have to add.

In [ ]:
cold_readers = set(data.cold_readers)


def cold_coverage(path):
    # The number of distinct zero-train books in the top-10 of ANY fitted, non-cold reader of `path`.
    surfaced = set()
    # TODO: readers = [int(r) for r in path.artifact()["reader_ids"] if int(r) not in cold_readers]
    # TODO: for each reader: for c in path.retrieve(reader, {"seen": data.seen_by.get(reader, set())}, K):
    # TODO:     if c.item_id in zero_train: surfaced.add(c.item_id)
    return len(surfaced)


feature_tower_cold = None   # TODO: cold_coverage(paths["feature-tower"])   -- 113 (of 818)
two_tower_cold = None       # TODO: cold_coverage(paths["two-tower"])       -- 0

feature_tower_lists = None  # TODO: system.top_lists(paths["feature-tower"])
blend_lists = None          # TODO: system.top_lists(system.fixed_blend())   (the first full-pool pass, ~20 s)
blend_without_ft_lists = None   # TODO: system.top_lists(system.fixed_blend(drop="feature-tower"))
stack_lists = None          # TODO: system.top_lists(system.full_stack)

cold_slice = None           # TODO: {"feature tower": len(cold_in(feature_tower_lists)),     -- 109
                            #        "fixed blend": len(cold_in(blend_lists)),                -- 23
                            #        "fixed blend without feature tower": len(cold_in(blend_without_ft_lists)),  -- 27
                            #        "full stack": len(cold_in(stack_lists))}                 -- 0
blend_cold_slots = None     # TODO: the number of (reader, slot) entries of blend_lists that are zero-train   -- 49
blend_cold_from_ft = None   # TODO: len(cold_in(blend_lists) & cold_in(feature_tower_lists))   -- 15

feature_tower_reaches_cold = None   # TODO: bool - feature_tower_cold > 0 and two_tower_cold == 0
stack_shows_no_cold = None          # TODO: bool - cold_slice["full stack"] == 0

cold_explanation = "TODO"  # 3-4 sentences: only the content paths (lexical, semantic, the hybrid's lexical
                           # half) and the feature tower can score a zero-train book; in the calibrated
                           # union a cold book usually has one vote against warm books with several, so few
                           # reach the top-10, and the lexical/semantic paths supply most of them (dropping
                           # the feature tower even frees a few slots); the reranker learned that read books
                           # are popular (log_pop, Exercise 9), so it ranks every cold book out; serving new
                           # books would need a reserved cold slot or a cold-aware ranker

## Exercise 3 - Hard negatives: the cached negative-pool ablation

A two-tower or feature tower learns by pushing a reader's vector **towards** books they read and **away from** sampled *negatives*. Which books are drawn as negatives is a design choice. The served feature tower uses `negative_pool="warm"` (uniform over warm books). The **hard** variant samples popular warm books more often -- harder negatives, the ones a reader most plausibly considered. Unit 9 measured the trade; the cache holds both towers, so you can read it here without a refit: `paths["feature-tower"]` (warm) and `paths["feature-tower-hard"]` (hard), both trained with seed 0 and the same recipe apart from the pool.

For both towers, read the full `system.evaluate` readout on the cohort and the cold coverage of Exercise 2. Expect (hit / coverage / head-share / exposure Gini / novelty): warm **0.298 / 0.307 / 0.638 / 0.877 / 8.94**, hard **0.172 / 0.450 / 0.275 / 0.777 / 10.25**, and cold coverage **113 vs 185**. Also read each tower's last training loss with `cache.loss_history(name)[-1]` (about 0.069 vs 0.071). In three or four sentences, state the trade with numbers on both sides, explain *why* popular negatives push the tower away from the head, say why the two final losses cannot be compared as "which tower is better", and say why the full stack serves the **warm** tower.

In [ ]:
hard_tower = paths["feature-tower-hard"]   # loaded from the cache -- do NOT refit a feature tower

warm_readout = None   # TODO: system.evaluate(paths["feature-tower"])
hard_readout = None   # TODO: system.evaluate(hard_tower)
negative_pool_table = None   # TODO: pd.DataFrame, one row per pool ("warm", "hard"): "hit", "coverage",
                             #       "head_share", "exposure_gini", "novelty", and "cold_coverage" (Exercise 2's
                             #       helper; reuse feature_tower_cold for warm)
                             #   -- warm 0.298 / 0.307 / 0.638 / 0.877 / 8.94 / 113
                             #      hard 0.172 / 0.450 / 0.275 / 0.777 / 10.25 / 185

final_losses = None   # TODO: {name: cache.loss_history(name)[-1] for name in ("feature-tower", "feature-tower-hard")}
                      #   -- ~0.069 and ~0.071

hard_costs_warm_hit = None   # TODO: bool - hard hit < warm hit
hard_widens_reach = None     # TODO: bool - hard coverage > warm coverage AND hard cold coverage > warm cold coverage
hard_leaves_the_head = None  # TODO: bool - hard head_share < warm head_share

hard_negative_explanation = "TODO"  # 3-4 sentences: hard negatives lose ~0.126 hit but add reach (coverage
                                    # 0.307 -> 0.450, cold 113 -> 185) and halve the head-share; sampling popular
                                    # books as negatives teaches the tower to score the head DOWN, so it spreads
                                    # toward the tail -- and a popular unread book may be a false negative; the two
                                    # losses are over different negative sets, so they measure different tasks;
                                    # the stack serves warm because its hit cost is too large for the system

## Exercise 4 - Learning to rank: a feature-family mask and refit on the cached training matrix

The full stack's reranker is a **learning-to-rank** model: for every candidate in a reader's 50-book pool it builds a 21-number **ranking-feature** vector and a small neural network scores it. The 21 features come in four **families**: `scores` (the 8 paths' calibrated scores), `presence` (8 flags: which paths proposed the book), `meta` (`n_paths`, the share of paths that proposed it) and `content` (`genre_frac`, `genre_cos`, `author_frac`, `log_pop`). `feature_names(FULL8_PATH_NAMES)` lists them in order.

The build script trained it with the Unit-11 **leakage-safe** recipe and cached its training matrix: `X, y = cache.training_matrix("reranker-full8")`. Check its shape (**7,458 x 21**), its positive rate (**about 0.091** -- one held-out positive per ten sampled negatives), and its recipe (`cache.meta("reranker-full8")["recipe"]` is `"u11-holdout-75-25"`). Because the matrix is cached, a refit is just the MLP, under a second.

First, refit on **all** features (`RerankerModel(seed=SEED).fit(X, y)`) and confirm every weight is `np.allclose` to the cached model's (`cache.artifact("reranker-full8")["model"]["weights"]`). Then refit with `feature_mask=family_mask(families, FULL8_PATH_NAMES)` for three variants: all four families, **content only**, and **no content** (`scores`, `presence`, `meta`). Serve each with the given `with_model` helper (a fresh reranker over the same 8-path pool) and record hit@10, coverage, head-share and the exposure Gini. Expect: all **0.356 / 0.2025 / 0.847 / 0.913** (exactly the full stack), content only **0.384 / 0.1365 / 0.960 / 0.940**, no content **0.354 / 0.186 / 0.845 / 0.920**. In three or four sentences, say which family carries the hit, what it costs, and why adding the path scores to the content features *dilutes* the content-only model.

In [ ]:
def with_model(model):
    # Given: a fresh full-stack reranker over the same 8 serving paths, with `model` as its ordering key.
    # (It loads the cached serving statistics; its refit factory is never called.)
    reranker = NeuralRerankerPath(
        system.full8, lambda rows: {}, data.catalog, pool=50, seed=SEED, path_names=FULL8_PATH_NAMES
    ).load(cache.artifact("reranker-full8"))
    return reranker.set_model(model)


X, y = None, None          # TODO: cache.training_matrix("reranker-full8")
matrix_shape = None        # TODO: X.shape                -- (7458, 21)
positive_rate = None       # TODO: float(y.mean())        -- ~0.091
recipe = None              # TODO: cache.meta("reranker-full8")["recipe"]   -- "u11-holdout-75-25"
names = None               # TODO: feature_names(FULL8_PATH_NAMES)          -- 21 names, content last

refit_model = None         # TODO: RerankerModel(seed=SEED).fit(X, y)
refit_matches_cache = None # TODO: bool - every np.allclose(a, b) for a, b in zip(refit_model.weights,
                           #       cache.artifact("reranker-full8")["model"]["weights"])

FAMILY_VARIANTS = {
    "all": ("scores", "presence", "meta", "content"),
    "content only": ("content",),
    "no content": ("scores", "presence", "meta"),
}
variant_models = None      # TODO: {name: RerankerModel(seed=SEED).fit(X, y, feature_mask=family_mask(families,
                           #        FULL8_PATH_NAMES)) for name, families in FAMILY_VARIANTS.items()}
family_table = None        # TODO: pd.DataFrame, one row per variant: "variant", "hit", "coverage", "head_share",
                           #       "exposure_gini" from system.evaluate(with_model(model))
                           #   -- all 0.356 / 0.2025 / 0.847 / 0.913, content only 0.384 / 0.1365 / 0.960 / 0.940,
                           #      no content 0.354 / 0.186 / 0.845 / 0.920

content_has_best_hit = None    # TODO: bool - "content only" has the highest hit of the three
content_is_narrowest = None    # TODO: bool - "content only" has the lowest coverage AND the highest head_share
scores_dilute_content = None   # TODO: bool - the "all" hit < the "content only" hit

ranking_explanation = "TODO"  # 3-4 sentences: the content family (genre/author affinity + log-popularity)
                              # carries the hit -- it sees a cross-reader taste signal the per-list calibrated
                              # path scores cannot; it buys that hit with reach (coverage 0.137, head-share
                              # 0.96); the path scores are correlated, noisy features that the MLP also leans on,
                              # so mixing them in pulls the ranking back toward the blend (dilution)

## Exercise 5 - Self-attention on the session log: a SASRec readout and an order probe

The sequence path is the one piece of the system scored on its **own** log: the session log, where each reader's reads are time-ordered sessions (design §6). The cached SASRec model (`system.sasrec`) is Unit 12's tiny causal **self-attention** block: it reads a reader's last <= 50 books in order, and the final position's hidden state -- which attends over every earlier position -- scores the next book. `system.session_top_lists()` gives its top-10 lists on the session log's val cohort, and `data.session_cohort` gives `(seen, relevant, cohort)` for that log.

First the readout. Report the session cohort size (**1,419** readers), SASRec's hit@10 (**about 0.423**) and catalog coverage (**about 0.285**), and its cached training curve (`cache.loss_history("sasrec")`: **30** epochs, about **7.30 -> 5.06**). Then put it next to a counting baseline: fit item-item CF on the **session** rows (`ItemItemRetrievalPath().fit(data.session_rows, catalog=data.catalog_ids)`, a cheap fit) and compute the paired difference SASRec minus item-item over the same readers, with its **paired standard error** `np.std(diffs, ddof=1) / np.sqrt(len(diffs))` (Unit 12). Expect item-item at **about 0.480**, so SASRec **trails** by about 0.057 (SE about 0.012).

Then probe what the attention actually uses -- **without refitting**. The cached artifact stores every reader's ordered history. The given `probe(edit)` copies the artifact with each stored history passed through `edit`, loads a fresh `SasrecRetrievalPath` from it (numpy only), and returns the new top-10 lists. Run three edits: **reversed** (`h[::-1]`), **last 1** (`h[-1:]`) and **last 5** (`h[-5:]`). Report each hit@10 and the paired drop (ordered minus edited) with its SE. Expect reversed **0.374** (a drop of about 0.049, SE about 0.010), last 1 **0.361** (about 0.062, SE about 0.013) and last 5 **0.429** (about -0.006, SE about 0.009 -- no loss). In three or four sentences, say what the probes show about what the model attends to, why reversing is a fair "does order matter?" probe here, and why these numbers must not be compared with main-log hit@10.

In [ ]:
session_seen, session_relevant, session_cohort = data.session_cohort


def session_hits(lists):
    # 1.0 / 0.0 per session-cohort reader: did their top-10 hit an unseen session-log val positive?
    return np.array([1.0 if set(lists[r]) & session_relevant[r] else 0.0 for r in session_cohort])


def paired(a, b):
    # (mean of a - b, paired standard error) over the same readers.
    diffs = a - b
    return float(diffs.mean()), float(np.std(diffs, ddof=1) / np.sqrt(len(diffs)))


sasrec_state = cache.artifact("sasrec")


def probe(edit):
    # Given: SASRec's top-10 lists after editing every stored history with `edit` -- no refit.
    state = dict(sasrec_state)
    state["histories"] = {r: edit(list(h)) for r, h in sasrec_state["histories"].items()}
    edited = SasrecRetrievalPath().load(state)
    return {r: top_k(edited, r, session_seen.get(r, set())) for r in session_cohort}


session_cohort_size = None   # TODO: len(session_cohort)                          -- 1419
sasrec_lists = None          # TODO: system.session_top_lists()
sasrec_hits = None           # TODO: session_hits(sasrec_lists)                   -- mean ~0.423
sasrec_coverage = None       # TODO: catalog_coverage(list(sasrec_lists.values()), data.catalog_ids)   -- ~0.285
sasrec_loss = None           # TODO: cache.loss_history("sasrec")                 -- 30 epochs, ~7.30 -> ~5.06

session_item_item = None     # TODO: ItemItemRetrievalPath().fit(data.session_rows, catalog=data.catalog_ids)
item_item_hits = None        # TODO: session_hits({r: top_k(session_item_item, r, session_seen.get(r, set()))
                             #                     for r in session_cohort})       -- mean ~0.480
sasrec_vs_item_item = None   # TODO: paired(sasrec_hits, item_item_hits)          -- ~(-0.057, 0.012)

EDITS = {"reversed": lambda h: h[::-1], "last 1": lambda h: h[-1:], "last 5": lambda h: h[-5:]}
probe_table = None           # TODO: pd.DataFrame, one row per edit: "edit", "hit" (session_hits(probe(edit)).mean()),
                             #       "drop" and "se" (paired(sasrec_hits, the edited hits))
                             #   -- reversed 0.374 / +0.049 / 0.010, last 1 0.361 / +0.062 / 0.013,
                             #      last 5 0.429 / -0.006 / 0.009

order_matters = None         # TODO: bool - the reversed drop > 2 * its SE
recent_window_suffices = None   # TODO: bool - abs(the last-5 drop) < 2 * its SE

attention_explanation = "TODO"  # 3-4 sentences: reversing the history (same books, new order) costs ~0.049 at
                                # ~5 SE, so the attention reads ORDER, not just the set; the last read alone
                                # loses more, but the last 5 lose nothing -- the model attends mostly to a short
                                # recent window; it still trails counting CF on this log; the session log is a
                                # different reader population and target, so its hit@10 is not on the main
                                # log's scale

## Exercise 6 - The ablation table: what each stage and each path buys

The build script wrote the system's **ablation table** to `readouts.json`, from freshly built caches, on every CI run. `cache.readouts["ablation"]` holds exactly 10 rows, each with its own `hit` and `coverage`, a declared `baseline`, and the signed deltas `d_hit` / `d_coverage` = row minus baseline. Read the baselines carefully, because they differ by row kind. The two **stage** rows (`no-reranker`, `exact-two-tower`) are deltas against the **full stack** (about 0.356 / 0.2025). The eight **path** rows (`minus-popularity`, ..., `minus-hybrid`) are **fixed-blend leave-one-out**: one path removed from the fixed 8-path blend, compared with the fixed blend (about 0.362 / 0.206) -- *not* reranked.

Put the 10 rows in a `DataFrame` (columns `row`, `kind`, `baseline`, `hit`, `coverage`, `d_hit`, `d_coverage`) and **recompute** every delta from `cache.readouts["baselines"]` (to 1e-9). Then recompute two rows **live** -- the fixed blend and `minus-popularity` (`system.fixed_blend(drop="popularity")`) -- and confirm they equal the cached rows exactly (expect **0.362 / 0.206** and **0.370 / 0.220**). Expect the stage rows `no-reranker` **+0.006 / +0.0035** and `exact-two-tower` **-0.002 / 0.000**. Among the path rows, find the removal that costs the most hit (expect **two-tower, -0.010**), the one that costs the most coverage (expect **lexical, -0.031**), and the removals that cost **neither** (expect **popularity** and **feature tower**). In three or four sentences, read the table: what does the reranker stage buy, what does ANN cost, and how much weight can a +-0.01 difference on 500 readers (5 readers) carry?

In [ ]:
readouts = cache.readouts   # the build's readouts.json (val only)

ablation = None           # TODO: pd.DataFrame([{key: row[key] for key in ("row", "kind", "baseline", "hit",
                          #       "coverage", "d_hit", "d_coverage")} for row in readouts["ablation"]])
deltas_recomputed = None  # TODO: bool - for every row, abs(d_hit - (hit - readouts["baselines"][baseline]["hit"]))
                          #       <= 1e-9, and the same for d_coverage / coverage

live_blend = None         # TODO: system.evaluate(system.fixed_blend())                   -- 0.362 / 0.206
live_minus_popularity = None   # TODO: system.evaluate(system.fixed_blend(drop="popularity"))   -- 0.370 / 0.220
live_matches_cache = None # TODO: bool - live_blend hit/coverage == readouts["baselines"]["fixed-blend"]'s, and
                          #       live_minus_popularity's == the "minus-popularity" row's (within 1e-9)

stage_rows = None         # TODO: the two "stage" rows of `ablation`   -- no-reranker +0.006 / +0.0035,
                          #                                              exact-two-tower -0.002 / 0.000
path_rows = None          # TODO: the eight "path" rows
costliest_hit = None      # TODO: the path row with the most negative d_hit       -- "minus-two-tower" (-0.010)
costliest_coverage = None # TODO: the path row with the most negative d_coverage  -- "minus-lexical" (-0.031)
free_removals = None      # TODO: sorted rows with d_hit >= 0 and d_coverage >= 0
                          #       -- ["minus-feature-tower", "minus-popularity"]

ablation_explanation = "TODO"  # 3-4 sentences: removing the reranker RAISES hit and coverage a little, so the
                               # learned stage does not earn its place here; swapping ANN for the exact scan
                               # moves hit by -0.002 (one reader) and reach by 0, so ANN costs nothing; the
                               # dense paths carry hit, the content paths
                               # carry reach; but every delta is <= 0.01 hit -- about 5 of 500 readers, well
                               # inside a hit's sampling noise -- so the table ranks stages only roughly, and
                               # the path rows are fixed-blend leave-one-out, not what the reranked stack does

## Exercise 7 - Beyond accuracy, the Part-1 benchmark, and the frontier

Hit@10 is one axis. `cache.readouts["beyond_accuracy"]` has exactly three rows -- the **full system**, the **best single path** (the Unit-10 hybrid) and the **Part-1 build** (the Unit-6 blend of popularity, lexical, item-item and MF) -- with hit and the five beyond-accuracy columns: coverage, head-share, exposure Gini (over the full catalog), novelty and intra-list diversity. Put it in a `DataFrame`. Expect (hit / coverage / head-share / Gini / novelty / diversity): full system **0.356 / 0.2025 / 0.847 / 0.913 / 8.57 / 0.857**, best single path **0.362 / 0.192 / 0.862 / 0.922 / 8.48 / 0.850**, Part-1 build **0.306 / 0.333 / 0.793 / 0.908 / 8.49 / 0.871**. Recompute the **Part-1 benchmark** live with `system.evaluate(system.part1_build())` and confirm it matches (expect **0.306 / 0.333**).

Now place five systems on the **hit vs coverage** plane: the Part-1 build, the best single path, the fixed blend, the full stack, and Exercise 4's content-only reranker. A system is **dominated** if another one is at least as good on both axes and strictly better on one. The **frontier** is the set of systems no other system dominates. Write `dominated(name, points)` and find the frontier. Expect exactly **Part-1 build, fixed blend, content-only reranker**: the full stack and the hybrid are both dominated by the fixed blend. In three or four sentences, describe the frontier as a menu (what each point is best at, and what it gives up), say what the beyond-accuracy columns add that coverage alone does not, and say who should choose a point on it.

In [ ]:
beyond = None             # TODO: pd.DataFrame(readouts["beyond_accuracy"])[["system", "hit", "coverage",
                          #       "head_share", "exposure_gini", "novelty", "diversity"]]
part1_live = None         # TODO: system.evaluate(system.part1_build())     -- 0.306 / 0.333
part1_matches = None      # TODO: bool - part1_live's hit and coverage equal the "part1-build" row's (within 1e-9)

points = None             # TODO: {name: (hit, coverage)} for "part1-build", "best-single-path" (from `beyond`),
                          #       "fixed-blend", "full-stack" (from readouts["baselines"]) and "content-reranker"
                          #       (Exercise 4's family_table "content only" row)


def dominated(name, points):
    # True if some OTHER system is >= on both hit and coverage and > on at least one.
    return None   # TODO: hit, cov = points[name]; any(h >= hit and c >= cov and (h > hit or c > cov)
                  #       for other, (h, c) in points.items() if other != name)


frontier = None           # TODO: [name for name in points if not dominated(name, points)]
                          #   -- ["part1-build", "fixed-blend", "content-reranker"]
full_stack_dominated = None   # TODO: bool - dominated("full-stack", points)          -- True
part1_widest_reach = None     # TODO: bool - Part-1 has the highest coverage AND the lowest exposure Gini in `beyond`

frontier_explanation = "TODO"  # 3-4 sentences: Part-1 reaches furthest (coverage 0.333, lowest Gini, most
                               # diverse) at the lowest hit; the fixed blend is the balanced point; the content
                               # reranker has the best hit (0.384) and is the most concentrated (head-share
                               # 0.96); the full stack adds machinery but sits inside the frontier; head-share,
                               # Gini, novelty and diversity say WHO gets the slots, which coverage alone hides;
                               # choosing a point is a product and values decision, not a metric's verdict

## Exercise 8 - Challenge: your own weighting of the 8-path blend, placed on the frontier

The fixed blend gives every path weight 1. `FixedBlendPath(system.full8, weights)` takes your own `{path: weight}` (missing paths default to 1.0) and reuses the same memoized pool, so each readout takes about a second. Unit 10 found that the dense paths deserve more weight; try that first. The **dense-heavy** weighting -- popularity 0.25, lexical 0.5, semantic 0.5, two-tower 1.5, hybrid 1.5, the rest 1.0 -- should score about **0.376 / 0.1835** (head-share **0.885**, Gini **0.926**). Add it to Exercise 7's `points` and confirm it joins the frontier.

Then design **your own** weighting with a stated goal (more reach, more hit, less head-share, ...), evaluate it, add it to `points`, and report whether it lands on the frontier. For example, a reach-leaning weighting -- popularity 0, MF 0.5, item-item 1.5, semantic 1.5, the rest 1.0 -- gives about **0.356 / 0.267**, which is on the frontier: it matches the full stack's hit with about 0.065 more coverage. In three or four sentences, state your goal, your numbers, and what this says about the value of a learned reranker *versus* a few hand-tuned blend weights on this data.

In [ ]:
DENSE_HEAVY = {**{name: 1.0 for name in FULL8_PATH_NAMES},
               "popularity": 0.25, "lexical": 0.5, "semantic": 0.5, "two-tower": 1.5, "hybrid": 1.5}
dense_heavy = None        # TODO: system.evaluate(FixedBlendPath(system.full8, DENSE_HEAVY))
                          #   -- 0.376 / 0.1835 / head-share 0.885 / Gini 0.926
points_with_dense = None  # TODO: {**points, "dense-heavy": (dense_heavy["hit"], dense_heavy["coverage"])}
dense_on_frontier = None  # TODO: bool - not dominated("dense-heavy", points_with_dense)

MY_WEIGHTS = None         # TODO: your own {path: weight} dict, with your goal in a comment
my_readout = None         # TODO: system.evaluate(FixedBlendPath(system.full8, MY_WEIGHTS))
my_on_frontier = None     # TODO: bool - not dominated("mine", {**points_with_dense, "mine": (hit, coverage)})

weights_explanation = "TODO"  # 3-4 sentences: your goal and numbers; dense-heavy buys hit (0.376) with reach
                              # (0.184), a reach-leaning blend buys coverage at the full stack's hit; a few
                              # hand-set weights move the system ALONG the frontier more than the learned
                              # reranker did -- on this data the reranker's extra machinery has to justify itself

## Exercise 9 - Challenge: one ranking feature closes the door on cold books

Exercise 2 found that the full stack shows **no** zero-train book at all. The suspect is the `log_pop` ranking feature, the book's log train-positive count scaled to [0, 1]: a zero-train book has `log_pop` exactly 0. First compare its mean in the cached training matrix for positives vs negatives (`X[y == 1, j]` vs `X[y == 0, j]`, where `j = names.index("log_pop")`): expect about **0.548 vs 0.420** -- held-out reads lean popular, so the model learns "popular ranks higher".

`family_mask` drops whole families, but `feature_mask` accepts any boolean column mask. Build a mask that keeps every feature **except** `log_pop`, refit on the cached `X, y`, serve it with `with_model`, and read hit@10, coverage, head-share and the cold slice (`cold_in`). Expect about **0.338 / 0.2385 / 0.793**, with **29** distinct zero-train books surfaced (against 0 for the full stack). In three or four sentences, describe the trade one feature controls, explain why a ranker trained on logged reads will keep learning popularity, and say whether you would ship the `log_pop`-free ranker -- and for which goal.

In [ ]:
log_pop_column = None     # TODO: names.index("log_pop")   (names from Exercise 4)   -- 20, the last column
log_pop_means = None      # TODO: (float(X[y == 1, log_pop_column].mean()), float(X[y == 0, log_pop_column].mean()))
                          #   -- ~(0.548, 0.420)

without_log_pop_mask = None   # TODO: np.array([name != "log_pop" for name in names])
without_log_pop = None    # TODO: with_model(RerankerModel(seed=SEED).fit(X, y, feature_mask=without_log_pop_mask))
without_log_pop_lists = None  # TODO: system.top_lists(without_log_pop)
without_log_pop_readout = None   # TODO: system.readout(without_log_pop_lists)   -- 0.338 / 0.2385 / head-share 0.793
without_log_pop_cold = None   # TODO: len(cold_in(without_log_pop_lists))       -- 29 (the full stack: 0)

reopens_cold = None       # TODO: bool - without_log_pop_cold > 0 and the full stack's cold count (Exercise 2) == 0
costs_hit = None          # TODO: bool - without_log_pop_readout["hit"] < the full stack's hit (0.356)

log_pop_explanation = "TODO"  # 3-4 sentences: dropping log_pop trades ~0.018 hit for ~0.036 coverage, a lower
                              # head-share and 29 cold books; held-out reads are drawn from a log where popular
                              # books were more visible, so any ranker trained on it learns popularity; whether
                              # to ship it depends on the goal -- reach and new authors, or hit